# Exercise 4 – Neural Network Classification of MMS Plasma Regions

**Companion notebook:** `04_cnn_plasma_classification.ipynb`

**Labelled dataset:** `data/data_cleaned.nc` (companion — as used in the lecture notebook)  
**Unlabelled dataset:** `data/ex1_cleaned_unlabelled.nc` (your March 2018 dataset from Exercise 1)

Work through the cells in order. Each exercise has a **Problem** statement in markdown followed by a code cell with `# YOUR CODE HERE`.


In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader, TensorDataset

from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from spacephyml.datasets.mms import SpectrumDataset

sns.set_theme()
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

figure_path = Path('./figures')
figure_path.mkdir(exist_ok=True)

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

NC_LABELLED   = 'data/data_cleaned.nc'
NC_UNLABELLED = 'data/ex1_cleaned_unlabelled.nc'

REGION_NAMES  = {0: 'Solar Wind', 1: 'Ion Foreshock', 2: 'Magnetosheath', 3: 'Magnetosphere'}
REGION_LABELS = list(REGION_NAMES.values())
REGION_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
N_CLASSES     = 4
N_BINS        = 32
N             = 66    # ≈ 5 minutes at 4.5 s cadence
SAMPLES       = 521   # windows per class (balanced)
BATCH_SIZE    = 32
TEST_SIZE     = 0.30

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

---

## Background

In the companion notebook a CNN was trained on raw 2-D spectrograms and achieved 99 % accuracy on the labelled test set. Here you will:

1. **Build your own CNN** on the same labelled dataset — you choose the architecture (within a suggested design space).
2. **Train an MLP** on PCA-compressed features as an alternative, and compare the two.
3. **Apply both trained networks** to your *unlabelled* March 2018 dataset from Exercise 1, and compare the predicted region distribution to the cluster distribution you found in Exercise 3.

The final step closes the loop on the Track A sequence: you started with raw unlabelled data, characterised its structure with PCA and clustering, and now you use a supervised model — trained entirely on a different labelled interval — to assign physical region labels to every window.

### Design space for the CNN (Exercise 4.2)

The companion notebook uses 2 convolutional layers (1→4 and 4→4 maps, 3×3 kernels) followed by max-pooling and two FC layers. You are free to vary:

| Hyperparameter | Companion value | Suggested range |
|---|---|---|
| Number of conv layers | 2 | 2 – 4 |
| Feature maps per layer | 4 | 4 – 32 |
| Kernel size | 3×3 | (3×3) – (5×5) |
| Pooling | MaxPool2d(2) | MaxPool2d(2) or AvgPool2d(2) |
| FC hidden size | 128 | 64 – 256 |
| Dropout rate | 0.3 | 0.0 – 0.5 |

Start close to the companion and make one or two changes — then justify them with the training curves.

---

## Part A — CNN on raw spectrograms

### Exercise 4.1 – Load and split the labelled dataset

Load `NC_LABELLED` twice:

- **2-D version** (`flatten=False`, `transform=preprocess_2d`) for the CNN.
- **Flat version** (`flatten=True`) for the MLP in Part B — apply `log10p1` as a numpy transform after loading.

Perform a 70/15/15 stratified split on *indices* (as in the companion notebook) and create `DataLoader` objects for the CNN path. Print the number of training and test samples and the class proportions.

> **Note**: As seen in the lecture, this dataset is small due to the grouping of samples into specrum windows. One way of increasing the amout of data is to create overlapping windiws. The `SpectrumDataset` supports this though the parameter `step`. However if you create overlapping windows you risk data leakage from the validation and test sets to the training set if you perform the standard stratified split, as the same data sample can be pressent in windows in both training and validation/test sets. How can you split the datasets without data leakage if you use overlapping windows?

In [ ]:
def preprocess_2d(X: np.ndarray) -> np.ndarray:
    """log10p1 scaling + add channel dimension for CNN input."""
    return np.log10(1. + X)[np.newaxis, :, :]

def preprocess_flat(X: np.ndarray) -> np.ndarray:
    """log10p1 scaling for flat (1-D) feature vectors."""
    return np.log10(1. + X)

# YOUR CODE HERE
# 1. Load ds_2d with flatten=False, transform=preprocess_2d, samples=SAMPLES.
# 2. Load ds_flat with flatten=True, samples=SAMPLES (no transform — apply
#    preprocess_flat to the numpy array after calling ds_flat[:]).
# 3. Perform a stratified 70/15/15 split on indices (use the labels from ds_2d).
# 4. Create Subset objects and DataLoaders for the CNN (batch_size=BATCH_SIZE,
#    shuffle=True for train, False for validation and test).
# 5. Print sample counts and per-class proportions for both splits.


---

### Exercise 4.2 – Design and build your CNN

Define a `MyCNN` class that inherits from `nn.Module`. Use the design space in the Background section as a guide. Requirements:

- Input shape: `(batch, 1, N, N_BINS)` — a single-channel 2-D spectrogram.
- At least two convolutional layers with ReLU activations.
- At least one pooling layer.
- At least one fully connected hidden layer.
- Output: `(batch, N_CLASSES)` — raw logits (do **not** apply softmax here; `CrossEntropyLoss` expects logits).
- The flat size after the convolutional stack must be computed automatically (use the dummy-forward trick from the companion notebook).

Instantiate your model, print its architecture, and report the number of trainable parameters. In the markdown cell below, write one or two sentences explaining your architectural choices.

*(Edit this cell — explain your architectural choices)*

Architecture choices: 

In [ ]:
# YOUR CODE HERE
# Define class MyCNN(nn.Module) with the requirements above.
# Instantiate: my_cnn = MyCNN(n_time=N, n_bins=N_BINS, n_classes=N_CLASSES).to(DEVICE)
# Print the model and its trainable parameter count.


---

### Exercise 4.3 – Train the CNN and plot training curves

Copy the `train_model` function from the companion notebook (it works for any `nn.Module`). Train your CNN for up to 300 epochs with `lr=1e-3` and `patience=20`.

Plot training and test loss and accuracy curves side-by-side (use the companion's `plot_history` function or write your own). Save the figure.

In the markdown cell below, answer:

1. How many epochs did early stopping trigger at?
2. Do the train and test curves track closely, or is there a visible gap? What does that tell you?
3. How does your CNN's final test accuracy compare to the companion's 99.0%? If it is lower, suggest one architectural or training change that might close the gap.

In [ ]:
# YOUR CODE HERE
# 1. Copy (or import) train_model from the companion notebook.
# 2. Train: history_cnn = train_model(my_cnn, train_loader, validation_loader, ...)
# 3. Plot training curves and save to figure_path / '04a_cnn_curves.png'.


*(Edit this cell — answer the three questions above)*

1. 
2. 
3. 

---

### Exercise 4.4 – Evaluate the CNN

Use the `evaluate` function from the companion notebook (or write your own) to obtain predictions on the test set. Report:

1. Overall accuracy.
2. A `classification_report` (precision, recall, F1 per class).
3. A row-normalised confusion matrix plot, saved to `figures/04a_cnn_confusion.png`.

Which pair of regions is most confused, and does it match the companion notebook's result?

In [ ]:
# YOUR CODE HERE
# 1. Collect predictions: y_pred_cnn = evaluate(my_cnn, test_loader)
# 2. Retrieve true labels: _, y_test = ds_test[:]
# 3. Print accuracy_score and classification_report.
# 4. Plot and save confusion matrix (reuse plot_confusion from companion, or write your own).


---

## Part B — MLP on PCA-compressed features

### Exercise 4.5 – Prepare PCA features and build the MLP

Using the **flat** dataset loaded in Exercise 4.1:

1. Fit `PCA(n_components=8)` on the training split only. Transform both train and test sets.
2. Wrap the resulting numpy arrays in `TensorDataset` + `DataLoader` objects.
3. Define a `MLP` class with the following fixed architecture:
   - Input: 8 features (PCA components)
   - Hidden layers: `Linear(8 → 64) → ReLU → Dropout(0.3) → Linear(64 → 32) → ReLU`
   - Output: `Linear(32 → N_CLASSES)` (raw logits)
4. Instantiate the MLP, print it, and report the trainable parameter count.

> **Why fit PCA only on the training set?** See the discussion in Exercise 3.1 and the companion notebook — fitting on the full dataset would leak test-set information into the PCA axes.

In [ ]:
# TensorDatasets and DataLoaders
def make_pca_loader(X_pca, y, shuffle):
    ds = TensorDataset(
        torch.tensor(X_pca, dtype=torch.float32),
        torch.tensor(y,     dtype=torch.long),
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)
    
N_PCS = 8
# YOUR CODE HERE
# 1. Apply the train/test index split from Exercise 4.1 to the flat array
#    X_flat (shape: n_samples × (N*N_BINS)) — already log10p1 scaled.
# 2. Fit PCA(n_components=N_PCS) on X_flat[idx_train] only.
# 3. Project: X_pca_train = pca.transform(X_flat[idx_train])
#             X_pca_test  = pca.transform(X_flat[idx_test])
# 4. Build TensorDatasets and DataLoaders (batch_size=BATCH_SIZE).
# 5. Define class MLP(nn.Module) with the architecture above.
# 6. Instantiate: mlp = MLP().to(DEVICE); print it and its parameter count.


---

### Exercise 4.6 – Train and evaluate the MLP

Train the MLP using the same `train_model` function (it is model-agnostic). Use `lr=1e-3` and `patience=20`.

Plot training curves and a confusion matrix (save both). Then fill in the comparison table below:

| Model | Input | Parameters | Test accuracy | SW/IF F1 |
|-------|-------|-----------|--------------|----------|
| Companion CNN | Raw spectrogram (66×32) | 223,040 | 99.0% | 0.98 |
| Your CNN | Raw spectrogram (66×32) | ? | ? | ? |
| MLP | 8 PCA components | ? | ? | ? |

In the markdown cell below, explain why the MLP is expected to score lower than the CNN on the Solar Wind / Ion Foreshock pair specifically.

In [ ]:
# YOUR CODE HERE
# 1. Train the MLP: history_mlp = train_model(mlp, train_loader_pca, test_loader_pca, ...)
# 2. Plot and save training curves to figure_path / '04a_mlp_curves.png'.
# 3. Collect predictions and print accuracy + classification_report.
# 4. Plot and save confusion matrix to figure_path / '04a_mlp_confusion.png'.


*(Edit this cell — fill in the table above and explain the SW/IF performance gap)*

Explanation: 

---

## Part C — Classify the unlabelled dataset

### Exercise 4.7 – Apply both networks to the March 2018 data

Load the unlabelled dataset (`NC_UNLABELLED`, 5-minute windows, all available samples). Apply both trained networks:

- **CNN path:** load with `flatten=False`, apply `preprocess_2d`, run through `my_cnn`.
- **MLP path:** load with `flatten=True`, apply `preprocess_flat`, project with your fitted PCA, run through `mlp`.

Collect the predicted class label for every window from each network. Do **not** retrain or fine-tune the models — use the weights from Exercises 4.3 and 4.6 directly.

> **Note:** The models were trained on Nov–Dec 2017 data. March 2018 is a different month but still Earth dayside, so the spectral fingerprints of each region should be similar. Any systematic differences you observe later are scientifically interesting.

In [ ]:
# YOUR CODE HERE
# ── CNN inference ─────────────────────────────────────────────────────────
# 1. Load NC_UNLABELLED with flatten=False, transform=preprocess_2d.
#    (No label_column — the dataset has no labels.)
# 2. Create a DataLoader (batch_size=BATCH_SIZE, shuffle=False).
# 3. Run my_cnn in eval mode and collect argmax predictions.
#    Store as: y_cnn_unlabelled  (1-D integer array, length = n_windows)

# ── MLP inference ─────────────────────────────────────────────────────────
# 4. Load NC_UNLABELLED with flatten=True.
# 5. Apply preprocess_flat and pca.transform (the PCA fitted in Exercise 4.5).
# 6. Wrap in a TensorDataset + DataLoader.
# 7. Run mlp in eval mode and collect argmax predictions.
#    Store as: y_mlp_unlabelled

# Print counts per predicted region for both models.


---

### Exercise 4.8 – Compare predicted region distributions

Plot a grouped bar chart showing the **fraction of windows** assigned to each region by:

- Your CNN
- Your MLP
- The best clustering from Exercise 3 (use the tentative labels you assigned there)

Use the same region order and colours as `REGION_COLORS`. Save the figure to `figures/04a_region_distribution.png`.

In the markdown cell below, answer:

1. Do the CNN and MLP agree on the dominant region for this interval? If not, which model do you trust more, and why?
2. How does the neural-network prediction compare to the cluster distribution from Exercise 3? Are the fractions broadly consistent, or is there a large discrepancy for any region?
3. The CNN and MLP were trained on Nov–Dec 2017. March 2018 is a different interval. What could cause a systematic shift in predicted region fractions between the two periods, even if the model is working correctly?

In [ ]:
# YOUR CODE HERE
# 1. Compute fractional distribution (count / total) for CNN, MLP,
#    and your Exercise 3 clustering (use the GMM or best-algorithm labels
#    with tentative physical label assignments from Exercise 3.7).
#
# 2. Create a grouped bar chart:
#    - x-axis: the four region names
#    - bars side-by-side: CNN | MLP | Clustering
#    - y-axis: fraction of windows (0–1)
#    - Add a legend and save to figure_path / '04a_region_distribution.png'.


*(Edit this cell — answer the three questions above)*

1. 
2. 
3. 

---

### Exercise 4.9 – Overlay CNN predictions on the orbit plot

Return to the GSE orbit plot from Exercise 1.2b. Re-plot MMS-1's trajectory in the XY plane, but this time colour each position point by the **CNN-predicted region** for the nearest window in time.

Use `REGION_COLORS` for the four regions and add a legend. Overlay the Shue et al. magnetopause and bow shock models as dashed lines (copy from Exercise 1.2b).

Save to `figures/04a_orbit_cnn_labels.png`.

> **Hint:** Match each ephemeris timestamp to the nearest window centre using `np.searchsorted` or a nearest-neighbour lookup on the window timestamps from the `SpectrumDataset`.

In [ ]:
from cdasws import CdasWs
import matplotlib.patches as mpatches

TRANGE = ['2018-03-01', '2018-03-31']
R_E    = 6371.2

cdas = CdasWs()
status, data = cdas.get_data(
    'MMS1_MEC_SRVY_L2_EPHT89Q', ['mms1_mec_r_gse'],
    TRANGE[0], TRANGE[1],
)
pos_gse  = data['mms1_mec_r_gse']
pos_time = data['Epoch']
x_gse = pos_gse[:, 0] / R_E
y_gse = pos_gse[:, 1] / R_E

# Window timestamps from the unlabelled dataset
win_times = # <dataset>.timestamps 

# For each ephemeris point, find the nearest window and reject any point
# that is more than 10 minutes from the nearest sample timestamp.
# Without this guard, ephemeris points inside data gaps (manoeuvres,
# instrument off-times, grid-drop intervals) get silently assigned the
# label of whatever window is nearest — which may be hours away.
MAX_GAP = np.timedelta64(10, "m")   # 10-minute tolerance

pos_time_ns  = pos_time.astype('datetime64[ns]')
win_times_ns = win_times.astype('datetime64[ns]')

# searchsorted gives the right insertion point of each ephemeris timestamp
# in the sorted window-timestamp array.
idx_r = np.searchsorted(win_times_ns, pos_time_ns)   # right insertion index

# Check both the window to the left and the one to the right;
# pick whichever centre is closer.
idx_l = (idx_r - 1).clip(0, len(win_times) - 1)
idx_r = idx_r.clip(0, len(win_times) - 1)

dt_l = np.abs(pos_time_ns - win_times_ns[idx_l])
dt_r = np.abs(pos_time_ns - win_times_ns[idx_r])

idx_nn = np.where(dt_l <= dt_r, idx_l, idx_r)   # index of nearest window
dt_nn  = np.minimum(dt_l, dt_r)                  # distance to nearest window

# Keep only ephemeris points within MAX_GAP of a sample; mark the rest -1.
valid_pos     = dt_nn <= MAX_GAP
region_at_pos = np.where(valid_pos, y_cnn_unlabelled[idx_nn], -1)

# YOUR CODE HERE
# 1. Scatter-plot x_gse vs y_gse coloured by predicted region.
#    Use REGION_COLORS; add Earth circle at origin (radius=1 R_E).
# 2. Overlay magnetopause and bow shock model curves (copy from Ex 1.2b).
# 3. Add a legend with region names and colours.
# 4. Save to figure_path / '04a_orbit_cnn_labels.png'.


---

## Summary

In this exercise you built two supervised classifiers on the labelled MMS dataset and applied them to your unlabelled March 2018 data:

- A **CNN** operating on raw 2-D spectrograms and uses the convolutional layers to learn the feature extraction from the spectrum images.
- An **MLP** operating on 8 PCA components is simpler and faster, but its linear feature compression discards non-linear spectral patterns.
- Applying the trained models to the unlabelled dataset is a **zero-shot transfer**: the models were never exposed to March 2018 data. The predicted region fractions depend on both model accuracy and the true physical conditions during the new interval.
- Overlaying CNN predictions on the GSE orbit provides the most direct sanity check: physically, the magnetosphere should appear near origin, the magnetosheath near the magnetopause boundary, and the solar wind at dayside apogee. If the spatial pattern is consistent with these expectations, the model is likely generalising correctly.
- The comparison between neural-network predictions and unsupervised cluster assignments (Exercise 4.8) quantifies the agreement between two completely different approaches to the same problem — a powerful cross-validation when ground truth is unavailable.

**Key principle:** A supervised model trained on a labelled interval can label new data from the same mission — but always sanity-check against physical context (orbit, plasma parameters, boundary models) before trusting the predictions.

### Further reading
- Olshevsky et al. (2021) — CNN for MMS plasma region classification
- Toy-Edens et al. (2024) — unsupervised classification for comparison: https://doi.org/10.1029/2024JA032431
- PyTorch transfer learning tutorial: https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html
